# 05 — Cohort Retention Analysis

**Purpose:** Track how well we retain customers over time by their signup cohort.  
**Key question:** Of the customers who first transacted in month X, how many are still active 3, 6, 12 months later?  

**Tables used:** `marts.mart_cohort_retention`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Retention heatmap
Each row is a cohort (month customers first transacted). Each column is months since first transaction. Color = % still active.

In [ ]:
cohort = q(f"""
    SELECT cohort_month, cohort_size, months_since_first, retention_pct
    FROM `{PROJECT}.marts.mart_cohort_retention`
    WHERE cohort_size >= 1000
    ORDER BY cohort_month, months_since_first
""")

if not cohort.empty:
    pivot = cohort.pivot(index='cohort_month', columns='months_since_first', values='retention_pct')
    pivot.index = pivot.index.astype(str).str[:7]

    fig = px.imshow(pivot, text_auto='.0f', aspect='auto',
                    color_continuous_scale='RdYlGn',
                    labels=dict(x='Months since first txn', y='Cohort', color='Retention %'),
                    title='Customer retention by cohort (%)')
    fig.update_layout(height=600)
    fig.show()
else:
    print('No cohort data found. Run the mart_cohort_retention SQL first.')

## 2. Retention curves
How does retention decay over time? Overlay the last 6 cohorts.

In [ ]:
if not cohort.empty:
    recent = cohort[cohort['cohort_month'].astype(str) >= cohort['cohort_month'].astype(str).unique()[-6]]

    fig = px.line(recent, x='months_since_first', y='retention_pct',
                  color=recent['cohort_month'].astype(str).str[:7],
                  title='Retention curves — last 6 cohorts',
                  labels={'months_since_first': 'Months since first txn', 'retention_pct': 'Retention %'})
    fig.update_layout(legend_title='Cohort')
    fig.show()

## 3. Cohort sizes over time
How many new customers are we acquiring each month?

In [ ]:
if not cohort.empty:
    sizes = cohort[cohort['months_since_first'] == 0][['cohort_month', 'cohort_size']].copy()
    sizes['cohort_month'] = sizes['cohort_month'].astype(str).str[:7]

    fig = px.bar(sizes, x='cohort_month', y='cohort_size',
                 title='New customers per month (cohort size)',
                 labels={'cohort_month': 'Month', 'cohort_size': 'New customers'})
    fig.show()

## 4. Key retention milestones
What % of customers survive to 1, 3, 6, and 12 months?

In [ ]:
if not cohort.empty:
    milestones = cohort[cohort['months_since_first'].isin([1, 3, 6, 12])]
    avg_retention = milestones.groupby('months_since_first')['retention_pct'].mean().reset_index()
    avg_retention.columns = ['months', 'avg_retention_pct']

    print('Average retention across all cohorts:\n')
    for _, row in avg_retention.iterrows():
        print(f'  {int(row["months"]):2d} months: {row["avg_retention_pct"]:.1f}%')

    fig = px.bar(avg_retention, x='months', y='avg_retention_pct',
                 title='Average retention at key milestones',
                 labels={'months': 'Months since first txn', 'avg_retention_pct': 'Avg retention %'},
                 text='avg_retention_pct')
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.show()

## 5. Best vs worst cohorts
Which cohorts retained the best at 6 months?

In [ ]:
if not cohort.empty:
    at_6m = cohort[cohort['months_since_first'] == 6][['cohort_month', 'cohort_size', 'retention_pct']].copy()
    at_6m['cohort_month'] = at_6m['cohort_month'].astype(str).str[:7]
    at_6m = at_6m.sort_values('retention_pct', ascending=False)

    print('Top 5 cohorts at 6-month retention:')
    display(at_6m.head())
    print(f'\nBottom 5 cohorts at 6-month retention:')
    display(at_6m.tail())
else:
    print('Not enough data for 6-month retention comparison.')